# 量子电路等价重写优化：快速上手

> 本 Notebook 用中文带你逐格理解重写版的核心 API：门库 → 电路 → 模拟器/语义等价 → 随机电路 → 观测张量 → 动作编码与掩码 → 4 条等价重写规则 → MDP 环境 → 策略网络 → PPO 的 GAE。

**运行前提**

1. 已按 `README.md` 第 4 节安装依赖（numpy、torch；可选 cirq）；
2. 建议在项目根目录启动 Jupyter（若在 `notebooks/` 目录启动，下面的代码会自动把项目根目录加入 `sys.path`）；
3. 本 Notebook 只演示核心逻辑，不会训练模型；完整训练/评估请使用 `scripts/train.py` 与 `scripts/evaluate.py`。

**阅读顺序**：直接从上到下执行即可。每个代码格都有中文注释与打印输出说明。

## 0. 准备：路径、依赖与导入

先用一小段代码定位项目根目录（保证 `import qco` 能找到包），然后一次性导入本 Notebook 需要的模块。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'qco').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch

from qco.circuit import Circuit, Op
from qco.gates import (
    GATE_NAMES,
    SINGLE_QUBIT_GATE_NAMES,
    TWO_QUBIT_GATE_NAMES,
    gate_matrix,
    is_unitary,
    match_single_qubit_gate,
)
from qco.simulator import circuit_unitary, circuits_equivalent
from qco.generator import random_circuit, random_circuits, save_circuits, load_circuits
from qco.observation import encode_circuit
from qco.actions import encode_action, decode_action
from qco.masking import build_action_mask, flatten_action_mask
from qco.rules import RULES, applicable_actions
from qco.environment import EnvConfig, CircuitOptimizationEnv
from qco.agent import CircuitPolicyNet
from qco.ppo import PPO, PPOConfig

print('Python', sys.version.split()[0])
print('numpy', np.__version__)
print('torch', torch.__version__)
print('CUDA 可用：', torch.cuda.is_available())
print('项目根目录：', ROOT)

## 1. 门库与门矩阵（`qco.gates`）

- 门库顺序固定：`X, Y, Z, H, S, Sdag, T, Tdag, CNOT, CZ, SWAP`；
- 单比特门是 2×2 幺正矩阵，二比特门是 4×4；比特 0 为最高位（big-endian）；
- `match_single_qubit_gate` 在**模全局相位**意义下把矩阵匹配到门库：`T·T -> S`、`H·H -> I`。

In [ ]:
print('门库顺序：', GATE_NAMES)
print('单比特门：', SINGLE_QUBIT_GATE_NAMES)
print('二比特门：', TWO_QUBIT_GATE_NAMES)

print('X =')
print(gate_matrix('X'))
print('CNOT =')
print(gate_matrix('CNOT'))

# 两个单比特门相乘后能否合并：T·T = S，H·H = I
print('T·T ->', match_single_qubit_gate(gate_matrix('T') @ gate_matrix('T')))
print('H·H ->', match_single_qubit_gate(gate_matrix('H') @ gate_matrix('H')))
print('H 矩阵是否幺正：', is_unitary(gate_matrix('H')))

## 2. 不可变电路表示（`qco.circuit`）

电路由若干时隙（layer / moment）组成，同一时隙内不同比特上的门互不相交。
`Circuit` 与 `Op` 都不可变；任何重写都会返回新电路。

下面构造一个简单电路：在 q0 上连续放两个 `T`，然后与只放一个 `S` 的电路比较——它们应该等价。

In [ ]:
c_tt = Circuit(2, [[Op('T', (0,))], [Op('T', (0,))]])
c_s = Circuit(2, [[Op('S', (0,))]])

print('电路：', c_tt)
print('深度：', c_tt.depth, ' 门数：', c_tt.num_ops, ' 代价 C = 门数 + 0.5×深度：', c_tt.cost(0.5))
print('ASCII 电路图：')
print(c_tt.to_ascii())
print('序列化（数据集 JSON 的单项格式）：', c_tt.to_dict())

## 3. 模拟器与语义等价（`qco.simulator`）

本项目的“优化”必须保持电路语义不变：比较两个电路的幺正矩阵是否**模全局相位**相等。

要点：全局相位不可观测，是量子电路等价的标准定义；`circuits_equivalent` 内部会计算 2ⁿ×2ⁿ 幺矩阵。

In [ ]:
u = circuit_unitary(c_tt)
print('T·T 电路的幺正矩阵形状：', u.shape)
print('是否幺正：', is_unitary(u))

# T·T 与 S 只差一个可能的全局相位（这里恰好完全相等）
print('T,T 与 S 语义等价：', circuits_equivalent(c_tt, c_s))

# 自逆门抵消示例：CNOT · X(目标比特) · CNOT = X(目标比特)
c1 = Circuit(2, [[Op('CNOT', (0, 1))], [Op('X', (1,))], [Op('CNOT', (0, 1))]])
c2 = Circuit(2, [[Op('X', (1,))]])
print('CNOT·X·CNOT 与 X 语义等价：', circuits_equivalent(c1, c2))

## 4. 随机电路与数据集（`qco.generator`）

训练集/评估集都是固定种子生成的随机电路：5 比特、深度 12、以概率 0.15 放二比特门。
数据集以 JSON 存取，便于复现实验。

In [ ]:
circuits = random_circuits(count=5, n_qubits=5, depth=12, seed=0)
print('第一个随机电路：', circuits[0])
print('门数：', circuits[0].num_ops, ' 深度：', circuits[0].depth, ' 代价：', circuits[0].cost(0.5))

dataset_path = ROOT / 'data' / 'quickstart_demo.json'
save_circuits(circuits, dataset_path)
loaded = load_circuits(dataset_path)
print('已写入并读回：', dataset_path)
print('读回数量：', len(loaded), ' 与原始第一个电路一致：', loaded[0] == circuits[0])

## 5. 观测张量：把电路编码成图像（`qco.observation`）

观测形状为 `[C, Q, M] = [14, 5, 15]`：

| 通道 | 含义 |
| --- | --- |
| 0–10 | 11 种门的 one-hot（二比特门在两个比特上同时为 1） |
| 11 | 二比特门的第一个比特（control） |
| 12 | 二比特门的第二个比特（target） |
| 13 | 该比特该时隙空闲（idle） |

超出电路实际深度的时隙全部标记为 idle，因此无论电路多浅，观测形状都保持不变。

In [ ]:
observation = encode_circuit(c_tt, n_moments=15)
print('观测张量形状 [C, Q, M]：', observation.shape)

channel_names = list(GATE_NAMES) + ['control', 'target', 'idle']
for channel, name in enumerate(channel_names):
    count = int(observation[channel].sum())
    if count > 0:
        print(f'通道 {channel:2d} {name:8s} 置 1 次数：{count}')

## 6. 动作编码与掩码（`qco.actions`、`qco.masking`）

- 动作是 `(规则 r, 比特 q, 时隙 m)`，统一编码为 `index = (r*Q + q)*M + m`；
- 动作掩码 `[R, Q, M]` 只在“规则可适用且时隙 < 当前深度”时为 `True`；
- 只有可行动作能被采样，非法动作概率严格为 0。

In [ ]:
index = encode_action(rule=1, qubit=2, moment=3, n_qubits=5, n_moments=15)
print('动作 (规则=1, 比特=2, 时隙=3) 编码为：', index)
print('解码回来：', decode_action(index, n_qubits=5, n_moments=15))

mask = build_action_mask(c_tt, n_moments=15)
print('掩码形状 [R, Q, M]：', mask.shape)
print('该电路上的可行动作数量：', int(flatten_action_mask(mask).sum()))

valid_flat = np.flatnonzero(flatten_action_mask(mask))
for flat_index in valid_flat:
    rule_index, qubit, moment = decode_action(int(flat_index), 5, 15)
    print('  动作下标', int(flat_index), '-> (规则', rule_index, ', 比特', qubit, ', 时隙', moment, ') 规则名：', RULES[rule_index].name)

## 7. 4 条等价重写规则（`qco.rules`）

规则顺序固定为：`merge_single_qubit`、`cancel_two_qubit`、`move_earlier`、`move_later`。

每条规则都满足两个不变量：**应用后代价严格下降**、**电路语义等价（模全局相位）**。
下面在 `T,T` 电路上应用“合并”规则：两个 T 合并为一个 S，代价从 3.0 降到 1.5。

In [ ]:
actions = applicable_actions(c_tt)
print('c_tt 上的所有可行动作 (rule, qubit, moment)：', actions)

rule_index, qubit, moment = actions[0]
new_circuit = RULES[rule_index].apply(c_tt, qubit, moment)
print('使用规则：', RULES[rule_index].name)
print('优化前代价：', c_tt.cost(0.5), ' -> 优化后代价：', new_circuit.cost(0.5))
print('优化前后语义等价：', circuits_equivalent(c_tt, new_circuit))

# 在默认随机电路上看看各规则分别有多少可用动作
from collections import Counter
random_circuit_demo = random_circuit(5, 12, np.random.default_rng(0))
rule_counts = Counter(RULES[r].name for r, _, _ in applicable_actions(random_circuit_demo))
print('随机电路上各规则的可行动作数：', dict(rule_counts))

## 8. MDP 环境（`qco.environment`）

环境把优化过程包装成 Gym 风格接口：

- `reset()` 返回初始观测；
- `step(action)` 返回 `(obs, reward, done, info)`；
- 奖励公式：`r = (代价_动作前 - 代价_动作后) / 初始代价`；
- `done` = 没有可用动作（terminated）或达到 `max_steps`（truncated）。

下面连续执行 5 步可行动作，验证奖励公式与“全程保持语义等价”。

In [ ]:
config = EnvConfig()
env = CircuitOptimizationEnv(config, circuit=random_circuit(5, 12, np.random.default_rng(0)))
env.reset()
print('初始代价：', env.cost, ' 初始门数：', env.circuit.num_ops, ' 初始深度：', env.circuit.depth)

total_reward = 0.0
for step in range(5):
    if not env.action_mask.any():
        print('没有可行动作，回合结束')
        break
    action = int(env.valid_actions[0])
    cost_before = env.cost
    _, reward, done, info = env.step(action)
    total_reward += reward
    rule_name = info['rule_name']
    expected = (cost_before - env.cost) / env.initial_cost
    print(f'第 {step + 1} 步 | 规则={rule_name:18s} | 奖励={reward:.4f}（公式 {expected:.4f}） | '
          f'代价 {cost_before:.1f} -> {env.cost:.1f} | done={done}')

print('累计奖励：', round(total_reward, 4))
print('前 5 步后仍与初始电路等价：', circuits_equivalent(env.initial_circuit, env.circuit))
print('info 字段示例：', sorted(info.keys()))

## 9. 策略网络前向（`qco.agent`）

网络是共享卷积主干的 Actor-Critic：

- 输入 `[B, 14, Q, M]`；
- 策略头输出 **logits** `[B, 4, Q, M]`（网络内不做 softmax）；
- 价值头输出 `[B]`；
- `act(obs, mask)` 内部只做一次 `masked_log_softmax`，返回 `(action, log_prob, value, entropy)`。

In [ ]:
policy = CircuitPolicyNet()
policy.eval()
print('网络参数量：', sum(p.numel() for p in policy.parameters()))

obs_tensor = torch.from_numpy(env.observation).float().unsqueeze(0)   # [1, C, Q, M]
mask_tensor = torch.from_numpy(env.action_mask).bool().unsqueeze(0)  # [1, R, Q, M]
with torch.no_grad():
    logits, value = policy(obs_tensor)
    action, log_prob, value, entropy = policy.act(obs_tensor, mask_tensor, deterministic=True)

print('logits 形状：', tuple(logits.shape))
print('value 形状：', tuple(value.shape))
print('确定性动作下标：', int(action.item()))
print('该动作的 log 概率：', round(float(log_prob.item()), 4))
print('策略熵（只统计合法动作）：', round(float(entropy.item()), 4))

## 10. PPO 的 GAE 小实验（`qco.ppo`）

GAE 是 PPO 的优势估计器，公式：

$$\delta_t = r_t + \gamma V(s_{t+1})(1-d_t) - V(s_t),\qquad A_t = \delta_t + \gamma\lambda(1-d_t)A_{t+1}$$

下面用一组手算可验证的小数据检查 `PPO.compute_gae` 的结果（注意最后一步也要包含并 bootstrap）。

In [ ]:
rewards = torch.tensor([[1.0, 2.0]])
values = torch.zeros(1, 2)
dones = torch.zeros(1, 2)
next_values = torch.tensor([3.0])

advantages, returns = PPO.compute_gae(
    rewards, values, dones, next_values, gamma=0.99, gae_lambda=0.95
)
print('优势 advantages：', advantages)
print('回报 returns = advantages + values：', returns)
print('手算校验：最后一步 delta = 2 + 0.99×3 = 4.97；前一步 A = 1 + 0.99×0.95×4.97 = 5.6743')
print('PPO 默认配置：', PPOConfig())

## 11. 下一步：训练与评估

本 Notebook 只演示核心组件；完整训练/评估请用命令行脚本（在项目根目录运行）：

```powershell
# 1) 生成评估集
python scripts/make_dataset.py --out data/eval.json --count 100 --n-qubits 5 --depth 12 --seed 10000

# 2) 训练（默认 150 轮）
python scripts/train.py --out results/default --iterations 150 --num-envs 8 --steps-per-iter 16 --seed 0 --device cpu

# 3) 评估（agent vs random / greedy）
python scripts/evaluate.py --checkpoint results/default/best.pt --dataset data/eval.json --out results/default/eval --baselines random,greedy

# 4) 生成论文插图
python scripts/make_figures.py --run results/default --out results/figures
```

如果 `python` 不在 PATH 中，使用虚拟环境的完整路径：

```powershell
C:\Users\zjc20\quantum-circuit-optimization\.venv\Scripts\python.exe scripts\train.py --out results\default --iterations 150
```

完整说明见项目根目录 `README.md`；实验设置与结果见 `docs/experiment.md`。

## 12. 小结与常见问题

本 Notebook 覆盖了写论文/答辩最常被问到的核心对象：

| 概念 | 代码位置 | 要点 |
| --- | --- | --- |
| 门库 | `qco/gates.py` | 11 个门 + 模全局相位匹配 |
| 电路 | `qco/circuit.py` | 不可变、`cost = 门数 + 0.5×深度` |
| 语义等价 | `qco/simulator.py` | 比较幺正矩阵，模全局相位 |
| 状态 | `qco/observation.py` | `[14, Q, M]` 张量，idle/padding |
| 动作 | `qco/actions.py` | `(r, q, m)` 单一整数编码 |
| 规则 | `qco/rules.py` | 4 条规则、严格降代价、语义等价 |
| 掩码 | `qco/masking.py` | 非法动作概率严格为 0 |
| 环境 | `qco/environment.py` | reset/step/势函数奖励 |
| 网络 | `qco/agent.py` | CNN 双头，输出 logits |
| PPO | `qco/ppo.py` | GAE、裁剪目标、KL 早停 |

**常见问题**

- `ModuleNotFoundError: No module named 'qco'`：请从项目根目录启动 Jupyter，或确认第 0 节的路径设置已执行；
- 没有安装 cirq：不影响本 Notebook（核心逻辑不依赖 cirq）；只有 `to_cirq()` / `to_text()` 的 Cirq 渲染需要它；
- 本 Notebook 创建了 `data/quickstart_demo.json`：这是演示文件，可随时删除；`data/*.json` 已在 `.gitignore` 中忽略。